In [ ]:
import pandas as pd
import json
import re

# -----------------------------
# INPUT / OUTPUT
# -----------------------------
csv_file = "../data/smmh.csv"
out_html = "../html-files/ster.html"

df = pd.read_csv(csv_file)

# -----------------------------
# COLUMNS
# -----------------------------
platform_col = "7. What social media platforms do you commonly use?"

question_cols = {
    "Q9": "9. How often do you find yourself using Social media without a specific purpose?",
    "Q10": "10. How often do you get distracted by Social media when you are busy doing something?",
    "Q11": "11. Do you feel restless if you haven't used Social media in a while?",
    "Q12": "12. On a scale of 1 to 5, how easily distracted are you?",
    "Q13": "13. On a scale of 1 to 5, how much are you bothered by worries?",
    "Q14": "14. Do you find it difficult to concentrate on things?",
    "Q15": "15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?",
    "Q16": "16. Following the previous question, how do you feel about these comparisons, generally speaking?",
    "Q17": "17. How often do you look to seek validation from features of social media?",
    "Q18": "18. How often do you feel depressed or down?",
    "Q19": "19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?",
    "Q20": "20. On a scale of 1 to 5, how often do you face issues regarding sleep?"
}

question_labels = {
    "Q9": "Purposeless use",
    "Q10": "Distracted while busy",
    "Q11": "Restless offline",
    "Q12": "Easily distracted",
    "Q13": "Bothered by worries",
    "Q14": "Concentration issues",
    "Q15": "Social comparison",
    "Q16": "Feeling after comparison",
    "Q17": "Validation seeking",
    "Q18": "Feeling down",
    "Q19": "Interest fluctuation",
    "Q20": "Sleep issues"
}

platforms = [
    "Instagram", "Facebook", "Twitter", "YouTube", "Discord",
    "Pinterest", "TikTok", "Snapchat", "Reddit", "LinkedIn"
]

# -----------------------------
# SUMMARISE DATA
# -----------------------------
rows = []

for platform in platforms:
    sub = df[
        df[platform_col]
        .fillna("")
        .str.contains(platform, case=False, regex=False)
    ]

    if sub.empty:
        continue

    values = {}
    for q, col in question_cols.items():
        values[q] = round(pd.to_numeric(sub[col], errors="coerce").mean(), 2)

    rows.append({
        "platform": platform,
        "n": int(len(sub)),
        "avgAge": round(pd.to_numeric(sub["1. What is your age?"], errors="coerce").mean(), 1),
        "occupation": sub["4. Occupation Status"].value_counts().to_dict(),
        "scores": values
    })

story_data = {
    "platforms": rows,
    "questions": [
        {
            "id": q,
            "short": question_labels[q],
            "full": question_cols[q]
        }
        for q in question_cols
    ]
}

DATA_JSON = json.dumps(story_data, indent=2)

# -----------------------------
# HTML
# -----------------------------
html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Was Grandma Right?</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,400;0,700;0,900;1,400;1,700&family=DM+Mono:wght@300;400;500&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>

<style>
:root {{
  --cream:#F5F0E8; --dark:#1A1209; --warm:#C8A96E;
  --accent:#E8472A; --muted:#6B5E4A; --card:#FFFDF7;
  --border:rgba(200,169,110,0.28);
}}

*{{box-sizing:border-box;margin:0;padding:0;}}

body{{
  background:var(--cream);
  color:var(--dark);
  font-family:'DM Sans',sans-serif;
  font-weight:300;
  line-height:1.7;
}}

.hero{{
  min-height:80vh;
  display:flex;
  flex-direction:column;
  justify-content:center;
  align-items:center;
  text-align:center;
  padding:4rem 2rem;
  position:relative;
}}

.hero::before{{
  content:'';
  position:absolute;
  inset:0;
  background:
    repeating-linear-gradient(0deg,transparent,transparent 39px,rgba(200,169,110,0.11) 39px,rgba(200,169,110,0.11) 40px),
    repeating-linear-gradient(90deg,transparent,transparent 39px,rgba(200,169,110,0.11) 39px,rgba(200,169,110,0.11) 40px);
  pointer-events:none;
}}

.eyebrow{{
  font-family:'DM Mono',monospace;
  font-size:11px;
  letter-spacing:.22em;
  text-transform:uppercase;
  color:var(--muted);
  margin-bottom:2rem;
  position:relative;
}}

h1{{
  font-family:'Playfair Display',serif;
  font-size:clamp(3rem,8vw,7rem);
  font-weight:900;
  line-height:1;
  max-width:950px;
  position:relative;
}}

h1 em{{color:var(--accent);}}

.hero-sub{{
  max-width:560px;
  color:var(--muted);
  margin-top:1.75rem;
  position:relative;
}}

section{{
  max-width:1100px;
  margin:0 auto;
  padding:5rem 2rem;
}}

.sec-label{{
  font-family:'DM Mono',monospace;
  font-size:10px;
  letter-spacing:.25em;
  text-transform:uppercase;
  color:var(--warm);
  margin-bottom:.9rem;
}}

h2{{
  font-family:'Playfair Display',serif;
  font-size:clamp(2rem,4vw,3rem);
  line-height:1.15;
  margin-bottom:.9rem;
}}

.lead{{
  max-width:680px;
  color:var(--muted);
  margin-bottom:2rem;
}}

.card{{
  background:var(--card);
  border:1px solid var(--border);
  border-radius:18px;
  padding:2rem;
  margin-bottom:1.5rem;
}}

.controls{{
  display:flex;
  flex-wrap:wrap;
  gap:.6rem;
  margin-bottom:1.8rem;
}}

.check{{
  font-family:'DM Mono',monospace;
  font-size:10px;
  letter-spacing:.08em;
  text-transform:uppercase;
  padding:.45rem .75rem;
  border:1px solid var(--border);
  border-radius:999px;
  background:rgba(255,253,247,.7);
  cursor:pointer;
}}

.check input{{margin-right:.35rem;}}

.chart-wrap{{
  position:relative;
  height:520px;
}}

.word-select{{
  font-family:'DM Mono',monospace;
  padding:.6rem .8rem;
  border:1px solid var(--border);
  border-radius:10px;
  background:var(--card);
  margin-bottom:1.5rem;
}}

.word-map{{
  min-height:360px;
  display:flex;
  flex-wrap:wrap;
  align-items:center;
  justify-content:center;
  gap:1rem 1.4rem;
  padding:2rem;
  border:1px dashed var(--border);
  border-radius:18px;
  background:rgba(200,169,110,0.08);
}}

.word {{
  font-family:'Playfair Display',serif;
  font-weight:900;
  line-height:1;
  transition:font-size .7s ease, opacity .7s ease, transform .7s ease, color .7s ease;
  cursor:default;
}}

.word:hover {{
  transform:scale(1.08) rotate(-1deg);
}}

.word-score {{
  display:block;
  font-family:'DM Mono',monospace;
  font-size:10px;
  font-weight:400;
  letter-spacing:.08em;
  text-align:center;
  margin-top:.35rem;
  color:var(--muted);
}}

@keyframes wordPop {{
  from {{
    opacity:0;
    transform:translateY(14px) scale(.9);
  }}
  to {{
    transform:translateY(0) scale(1);
  }}
}}

.stat-row{{
  display:grid;
  grid-template-columns:repeat(auto-fit,minmax(170px,1fr));
  gap:12px;
  margin-top:1rem;
}}

.stat{{
  background:var(--card);
  border:1px solid var(--border);
  border-radius:14px;
  padding:1.3rem;
}}

.stat-num{{
  font-family:'Playfair Display',serif;
  font-size:2.3rem;
  font-weight:900;
  color:var(--accent);
}}

.stat-label{{
  font-family:'DM Mono',monospace;
  font-size:9px;
  letter-spacing:.15em;
  text-transform:uppercase;
  color:var(--muted);
}}

.story-box{{
  background:var(--dark);
  color:var(--cream);
  border-radius:22px;
  padding:3rem;
  margin-top:2rem;
}}

.story-box h3{{
  font-family:'Playfair Display',serif;
  font-size:2rem;
  margin-bottom:1rem;
}}

.story-box p{{
  color:rgba(245,240,232,.72);
  max-width:760px;
}}
</style>
</head>

<body>

<div class="hero">
  <p class="eyebrow">Social media platforms · questions 9–20</p>
  <h1>Was <em>grandma</em> right?</h1>
  <p class="hero-sub">
    Choose one or more platforms and compare how users score across purposeless use,
    distraction, worries, validation seeking, mood, and sleep.
  </p>
</div>

<section>
  <p class="sec-label">01 — Platform comparison</p>
  <h2>Which platforms are linked to which feelings?</h2>
  <p class="lead">
    Select apps below. Each line shows the average score from 1 to 5 for users of that platform.
    Higher means the behaviour or feeling was reported more often.
  </p>

  <div class="card">
    <div class="controls" id="platformControls"></div>
    <div class="chart-wrap">
      <canvas id="scoreChart"></canvas>
    </div>
  </div>
</section>

<section>
  <p class="sec-label">02 — Strongest platform signal</p>
  <h2>Which platform speaks loudest?</h2>
  <p class="lead">
    Pick a question and watch the platforms resize. Bigger words mean higher average scores.
  </p>

  <div class="card">
    <select id="wordQuestionSelect" class="word-select" onchange="updateWordMap()"></select>
    <div id="wordMap" class="word-map"></div>
  </div>
</section>

<section>
  <p class="sec-label">03 — Instagram profile</p>
  <h2>Who are the Instagram users?</h2>
  <p class="lead">
    This section gives you the numbers for the discussion: age, validation seeking,
    and whether the users are mostly students, workers, or another group.
  </p>

  <div class="stat-row" id="instagramStats"></div>

  <div class="story-box">
    <h3>Discussion angle</h3>
    <p id="discussionText"></p>
  </div>
</section>

<script>
const D = {DATA_JSON};

const QUESTION_COLORS = {{
  Q9:  "#E8472A",
  Q10: "#C8A96E",
  Q11: "#534AB7",
  Q12: "#185FA5",
  Q13: "#D4537E",
  Q14: "#3B6D11",
  Q15: "#BA7517",
  Q16: "#888780",
  Q17: "#0F6E56",
  Q18: "#A64253",
  Q19: "#6F4E37",
  Q20: "#2F4858"
}};

const PLATFORM_COLORS = [
  "#E8472A","#C8A96E","#534AB7","#185FA5","#D4537E",
  "#3B6D11","#BA7517","#888780","#0F6E56","#A64253"
];

Chart.defaults.font.family = "'DM Mono', monospace";
Chart.defaults.color = "#6B5E4A";

const qIDs = D.questions.map(q => q.id);
const qLabels = D.questions.map(q => q.short);

const platformControls = document.getElementById("platformControls");

D.platforms.forEach((p) => {{
  const checked = ["Instagram", "TikTok", "YouTube"].includes(p.platform) ? "checked" : "";
  platformControls.innerHTML += `
    <label class="check">
      <input type="checkbox" value="${{p.platform}}" ${{checked}} onchange="updateChart()">
      ${{p.platform}}
    </label>
  `;
}});

const scoreChart = new Chart(document.getElementById("scoreChart"), {{
  type: "line",
  data: {{
    labels: qLabels,
    datasets: []
  }},
  options: {{
    responsive: true,
    maintainAspectRatio: false,
    plugins: {{
      legend: {{ position: "bottom" }},
      tooltip: {{
        callbacks: {{
          title: items => {{
            const idx = items[0].dataIndex;
            return D.questions[idx].id + " — " + D.questions[idx].short;
          }},
          afterTitle: items => {{
            const idx = items[0].dataIndex;
            return D.questions[idx].full;
          }},
          label: item => `${{item.dataset.label}}: ${{item.parsed.y.toFixed(2)}} / 5`
        }}
      }}
    }},
    scales: {{
      y: {{
        min: 1,
        max: 5,
        ticks: {{ callback: value => value + "/5" }},
        grid: {{ color: "rgba(200,169,110,0.15)" }}
      }},
      x: {{
        ticks: {{
          maxRotation: 55,
          minRotation: 35,
          color: ctx => QUESTION_COLORS[qIDs[ctx.index]]
        }},
        grid: {{ display: false }}
      }}
    }}
  }}
}});

function updateChart() {{
  const selected = [...document.querySelectorAll("#platformControls input:checked")]
    .map(x => x.value);

  scoreChart.data.datasets = D.platforms
    .filter(p => selected.includes(p.platform))
    .map((p) => {{
      const col = PLATFORM_COLORS[
        D.platforms.findIndex(x => x.platform === p.platform) % PLATFORM_COLORS.length
      ];

      return {{
        label: `${{p.platform}} (n=${{p.n}})`,
        data: qIDs.map(q => p.scores[q]),
        borderColor: col,
        backgroundColor: col + "33",
        pointBackgroundColor: qIDs.map(q => QUESTION_COLORS[q]),
        pointBorderColor: "#1A1209",
        pointRadius: 5,
        pointHoverRadius: 8,
        tension: 0.25,
        borderWidth: 2
      }};
    }});

  scoreChart.update();
}}

function makeWordQuestionSelect() {{
  const select = document.getElementById("wordQuestionSelect");

  select.innerHTML = D.questions.map(q => `
    <option value="${{q.id}}">
      ${{q.id}} — ${{q.short}}
    </option>
  `).join("");

  select.value = "Q17";
}}

function updateWordMap() {{
  const q = document.getElementById("wordQuestionSelect").value;
  const wordMap = document.getElementById("wordMap");

  const values = D.platforms.map(p => p.scores[q]);
  const minVal = Math.min(...values);
  const maxVal = Math.max(...values);

  const sorted = [...D.platforms].sort(
    (a, b) => b.scores[q] - a.scores[q]
  );

  wordMap.innerHTML = sorted.map((p, i) => {{
    const score = p.scores[q];

    const size =
      maxVal === minVal
        ? 42
        : 24 + ((score - minVal) / (maxVal - minVal)) * 58;

    const opacity =
      maxVal === minVal
        ? 1
        : 0.45 + ((score - minVal) / (maxVal - minVal)) * 0.55;

    const color = PLATFORM_COLORS[
      D.platforms.findIndex(x => x.platform === p.platform) % PLATFORM_COLORS.length
    ];

    return `
      <div class="word" style="
        font-size:${{size}}px;
        color:${{color}};
        opacity:${{opacity}};
        animation: wordPop .55s ease both;
        animation-delay:${{i * 60}}ms;
      ">
        ${{p.platform}}
        <span class="word-score">${{score.toFixed(2)}} / 5</span>
      </div>
    `;
  }}).join("");
}}

function makeInstagramStats() {{
  const insta = D.platforms.find(p => p.platform === "Instagram");
  if (!insta) return;

  const occEntries = Object.entries(insta.occupation)
    .sort((a,b) => b[1] - a[1]);

  const topOcc = occEntries[0][0];
  const topOccN = occEntries[0][1];
  const topOccPct = Math.round(topOccN / insta.n * 100);

  document.getElementById("instagramStats").innerHTML = `
    <div class="stat">
      <div class="stat-num">${{insta.n}}</div>
      <div class="stat-label">Instagram users</div>
    </div>
    <div class="stat">
      <div class="stat-num">${{insta.avgAge}}</div>
      <div class="stat-label">Average age</div>
    </div>
    <div class="stat">
      <div class="stat-num">${{insta.scores.Q17.toFixed(2)}}</div>
      <div class="stat-label">Validation score</div>
    </div>
    <div class="stat">
      <div class="stat-num">${{topOccPct}}%</div>
      <div class="stat-label">${{topOcc}}</div>
    </div>
  `;

  document.getElementById("discussionText").innerHTML = `
    Instagram users in this dataset have an average age of <strong>${{insta.avgAge}}</strong>.
    The largest occupation group is <strong>${{topOcc}}</strong>, making up around
    <strong>${{topOccPct}}%</strong> of Instagram users.
    Their average validation-seeking score is <strong>${{insta.scores.Q17.toFixed(2)}} / 5</strong>.
    This gives you a useful discussion point: are Instagram users especially associated with
    validation seeking, or does another platform show the same or stronger pattern?
  `;
}}

updateChart();
makeWordQuestionSelect();
updateWordMap();
makeInstagramStats();
</script>

</body>
</html>
"""

with open(out_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved: {out_html}")